## Program for creating database for storing observation used in RAG

Organisation of notebook are as follows:

- [1. Set up imports](#setup)
- [2. Choose DB collection to create](#collection)
- [3. Set up path for saving](#path)
- [4. Set up embeddings model to use](#embed)
- [5a. Create embeddings for class names collection](#class)
- [5b. Create embeddings for observation collection](#obs)
    

<a id="setup"></a>
## 1. Set up imports

In [ ]:
import os
import pandas as pd
import s3fs

import chromadb
from sentence_transformers import SentenceTransformer
from klass import get_classification                       # Note: package name is ssb-klass-python

from utils.formatting import reduce_test, get_sn
from utils.db_creation import index_classes, index_observations, upload_chroma_to_s3
from config import embeddings_model, chromadb_path, collection_name_class, collection_name_obs, train_path

<a id="collection"></a>
## 2. Choose which DB collection to create

Here we can choose between:

- 'obs': DB with embedded observations from a training dataset
- 'class': DB with embedded descriptions of the NACE class groups
- 'both': Run both of the above, but in seperate sections of the DB

In [ ]:
db_type = "class" # 'obs','class' or 'both'

<a id="path"></a>
## 3. Set up path to save the db

In [ ]:
S3_BUCKET = "projet-aiml4os-wp10/Cluster2/Norway-RAG-data"
S3_PREFIX = "chroma_storage"
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

if not os.path.exists(os.path.join(chromadb_path, "chroma.sqlite3")):
    download_chroma_from_s3(S3_BUCKET, S3_PREFIX, chromadb_path, fs)
else:
    print("Chroma already exists → skipping download")  

# Create a persistent ChromaDB client (saves to disk)
client = chromadb.PersistentClient(path=chromadb_path)

<a id="embed"></a>
## 4. Set up embedding model

In [ ]:
embedder = SentenceTransformer(embeddings_model)

<a id="class"></a>
## 5a. Create embeddings for class names collection

In [ ]:
if db_type in ("class", "both"):
    try:
        client.delete_collection(collection_name_class) # Delete collection if it exists
    except chromadb.errors.NotFoundError:
        pass
    
    # Create (or load) a collection
    collection_class = client.get_or_create_collection(
        name=collection_name_class,
        metadata={"hnsw:space": "cosine"}  # Use cosine similarity
    )

    # Get Description codes from SSBs klassification database using API
    sn = get_sn()

    # Embed description and save to DB
    index_classes(sn, code_col = "code", name_col = "name", note_col = "notes", embedder = embedder, collection = collection_class)

    # Upload collection to SSPcloud
    upload_chroma_to_s3(chromadb_path, S3_BUCKET, S3_PREFIX, fs)

<a id="obs"></a>
## 5b. Create embeddings for option with observations

In [ ]:
if db_type in ("obs", "both"):
    # Read in training data
    train = pd.read_parquet(train_path)
    train = reduce_test(train, min_nr = 2, state = 42, test_size = 50000) # reduce_test function is used to reduce the training data.
    print(f'Training data has shape {train.shape}')

    # Set up collection 
    try:
        client.delete_collection(collection_name_obs)
    except:
        pass

    # Create (or load) a collection
    collection_obs = client.get_or_create_collection(
        name=collection_name_obs,
        metadata={"hnsw:space": "cosine"}  # Use cosine similarity
    )

    # Embed observations and save to DB
    index_observations(train, 
                       embedder = embedder, 
                       collection = collection_obs)
    
    # Upload collection to SSPcloud
    upload_chroma_to_s3(chromadb_path, S3_BUCKET, S3_PREFIX, fs)